# Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

# Read Bronze table

In [0]:
df = spark.table("workspace.bronze.erp_px_cat_g1v2")

In [0]:
df.display()

# Silver Transformations

## Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

## Normalize Maintenance Flag to Boolean

In [0]:
df = df.withColumn(
    "MAINTENANCE",
    F.when(col("MAINTENANCE") =="Yes", F.lit(True))
    .when(col("MAINTENANCE") =="No", F.lit(False))
    .otherwise(None)
)

In [0]:
df.display()

## Renaiming columns

In [0]:
RENAME_MAP = {
    "ID": "category_id",
    "CAT": "category",
    "SUBCAT": "subcategory",
    "MAINTENANCE": "maintenance_flag"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

# Writing Silver table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.erp_product_category")

In [0]:
%sql
SELECT *
FROM silver.erp_product_category